In [1]:
"""
Programmatic Flip Rate Reduction Test
Simulate original inconsistency and test if QDs reduce flip rate
"""

import time
import json
import csv
import random
from collections import defaultdict
from datetime import datetime

# API Configuration
BASE_URL = "http://ece-nebula16.eng.uwaterloo.ca:11434"

In [2]:
def generate_with_delay(prompt: str, reasoning: bool = False, delay: float = 1.0) -> str:
    """Call LLM API with exponential backoff delay"""
    import requests
    
    payload = {
        "model": "gpt-oss:120b",
        "prompt": prompt,
        "stream": False
    }
    
    if delay > 0:
        time.sleep(delay)
    
    try:
        response = requests.post(
            f"{BASE_URL}/api/generate", 
            headers={"Content-Type": "application/json"},
            json=payload,
            timeout=60
        )
        
        if response.status_code == 200:
            result = response.json()
            return result.get("response", "No response returned")
        else:
            print(f"API Error: {response.status_code} - {response.text}")
            return "API Error occurred"
    except Exception as e:
        print(f"Connection Error: {e}")
        return "Connection Error occurred"

In [3]:
import pandas as pd

print("Loading problematic data:")
initial_df = pd.read_csv('./bad_questions.csv', delimiter='\t')

initial_df.head(5)

initial_df.columns


Loading problematic data:


Index(['username', 'lab_number', 'question_number', 'ts', 'question_text',
       'grade', 'term'],
      dtype='object')

In [4]:
# Group by lab number and username 
lab_1_df = initial_df[initial_df['lab_number'] == 1].copy()
lab_2_df = initial_df[initial_df['lab_number'] == 2].copy()
lab_3_df = initial_df[initial_df['lab_number'] == 3].copy()
lab_4_df = initial_df[initial_df['lab_number'] == 4].copy()
lab_5_df = initial_df[initial_df['lab_number'] == 5].copy()


print(lab_1_df.info())



<class 'pandas.core.frame.DataFrame'>
Index: 430 entries, 0 to 1735
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   username         430 non-null    object
 1   lab_number       430 non-null    int64 
 2   question_number  430 non-null    int64 
 3   ts               430 non-null    object
 4   question_text    430 non-null    object
 5   grade            430 non-null    int64 
 6   term             430 non-null    object
dtypes: int64(3), object(4)
memory usage: 26.9+ KB
None


In [5]:
def calculate_flip_rate(df):
    """
    Calculate flip rate: percentage of unique question_text entries with grade disagreements
    across submissions.
    """
    flip_count = 0
    total_questions = 0
    
    grouped = df.groupby('question_text')
    
    for question_text, group in grouped:
        if len(group) > 1:  # Only consider questions with multiple submissions
            total_questions += 1
            if group['grade'].nunique() > 1:  # Check for grade disagreements
                flip_count += 1
    
    flip_rate = (flip_count / total_questions * 100) if total_questions > 0 else 0
    
    return {
        'flip_count': flip_count,
        'total_questions': total_questions,
        'flip_rate': flip_rate
    }

# Calculate flip rates
initial_stats = calculate_flip_rate(initial_df)
lab_1_stats = calculate_flip_rate(lab_1_df)
lab_2_stats = calculate_flip_rate(lab_2_df)
lab_3_stats = calculate_flip_rate(lab_3_df)
lab_4_stats = calculate_flip_rate(lab_4_df)
lab_5_stats = calculate_flip_rate(lab_5_df)
print(f"All Labs: {initial_stats['flip_rate']:.2f}% ({initial_stats['flip_count']}/{initial_stats['total_questions']})")
print(f"Lab 1: {lab_1_stats['flip_rate']:.2f}% ({lab_1_stats['flip_count']}/{lab_1_stats['total_questions']})")
print(f"Lab 2: {lab_2_stats['flip_rate']:.2f}% ({lab_2_stats['flip_count']}/{lab_2_stats['total_questions']})")
print(f"Lab 3: {lab_3_stats['flip_rate']:.2f}% ({lab_3_stats['flip_count']}/{lab_3_stats['total_questions']})")
print(f"Lab 4: {lab_4_stats['flip_rate']:.2f}% ({lab_4_stats['flip_count']}/{lab_4_stats['total_questions']})")
print(f"Lab 5: {lab_5_stats['flip_rate']:.2f}% ({lab_5_stats['flip_count']}/{lab_5_stats['total_questions']})")



All Labs: 100.00% (379/379)
Lab 1: 100.00% (93/93)
Lab 2: 100.00% (117/117)
Lab 3: 100.00% (54/54)
Lab 4: 100.00% (72/72)
Lab 5: 100.00% (43/43)


In [9]:
# Create a dictionary mapping (lab_number, question_number) to rubrics
rubrics_dict = {}

# Read each rubric file and parse the rubrics
rubric_files = {
    1: './Rubrics/rubric_lab1 1.txt',
    2: './Rubrics/rubric_lab2 1.txt',
    3: './Rubrics/rubric_lab3 1.txt',
    4: './Rubrics/rubric_lab4 1.txt',
    5: './Rubrics/rubric_lab5 1.txt'
}

for lab_num, filename in rubric_files.items():
    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()
        # Split by "QUESTION" to get individual question blocks
        questions = content.split('QUESTION\n')[1:]  # Skip first empty split
        
        for idx, question_block in enumerate(questions, start=1):
            # Extract the rubric portion (everything after "The rubric is:")
            parts = question_block.split('The rubric is:')
            if len(parts) > 1:
                rubric = parts[1].strip()
                rubrics_dict[(lab_num, idx)] = rubric

print(f"Loaded {len(rubrics_dict)} rubrics")

# Add rubric column to the dataframe
initial_df['rubric'] = initial_df.apply(
    lambda row: rubrics_dict.get((row['lab_number'], row['question_number']), ''), 
    axis=1
)

# Check how many rows got rubrics
rows_with_rubric = (initial_df['rubric'] != '').sum()
print(f"Rows with rubric: {rows_with_rubric} out of {len(initial_df)}")
print(f"\nSample rubric for Lab 1, Question 1:")
print(initial_df[(initial_df['lab_number'] == 1) & (initial_df['question_number'] == 1)]['rubric'].iloc[0])


Loaded 25 rubrics
Rows with rubric: 1744 out of 1744

Sample rubric for Lab 1, Question 1:
Meets Expectation: Students correctly identify that the HAL is the Hardware Abstraction Layer. Students correctly identify that the HAL is used to simplify interaction with the hardware. Studens provide some additional, correct details regarding why it is useful to have a HAL. 

Does not Meet Expectation: The student work either contains errors, omissions, or is not written with sufficient detail.


In [11]:
print(initial_df[(initial_df['lab_number'] == 1) & (initial_df['question_number'] == 1)]['rubric'].unique())

['Meets Expectation: Students correctly identify that the HAL is the Hardware Abstraction Layer. Students correctly identify that the HAL is used to simplify interaction with the hardware. Studens provide some additional, correct details regarding why it is useful to have a HAL. \n\nDoes not Meet Expectation: The student work either contains errors, omissions, or is not written with sufficient detail.']


In [84]:
# Helper function: Extract quality dimensions from rubric
def extract_qds_from_rubric(rubric_text):
    """
    Extract quality dimensions from a rubric text using LLM.
    Returns list of QDs as dictionaries with 'name' and 'definition' keys.
    """
    print("  Calling LLM to extract QDs from rubric...")
    
    prompt = f"""Analyze this grading rubric and extract 3-5 binary quality dimensions.

RUBRIC:
{rubric_text}

A quality dimension should be:
- BINARY (present or absent, not a scale)
- DISTINGUISHABLE (can be reliably detected by reading the text)
- VARIABLE (present in some correct answers, absent in others)
- SPECIFIC to the rubric criteria (not generic like "clarity" or "completeness")

Focus on CONCEPTUAL criteria from the rubric, not writing style.

Output ONLY a valid JSON array with NO additional text:
[
  {{
    "name": "short_dimension_name",
    "definition": "clear definition of what presence means"
  }}
]"""
    
    response = generate_with_delay(prompt, reasoning=False, delay=1.5)
    
    # Parse JSON response
    try:
        # Try to extract JSON array from response
        import re
        json_match = re.search(r'\[.*\]', response, re.DOTALL)
        if json_match:
            qds = json.loads(json_match.group(0))
            if isinstance(qds, list) and len(qds) >= 2:
                print(f"  Successfully extracted {len(qds)} QDs")
                return qds[:5]  # Limit to 5 QDs
    except Exception as e:
        print(f"  Failed to parse QDs: {e}")
    
    # Fallback QDs based on common rubric patterns
    print("  Using fallback QDs")
    return [
        {"name": "correct_identification", "definition": "correctly identifies the main concept or term"},
        {"name": "explains_purpose", "definition": "explains why or how the concept is used"},
        {"name": "provides_details", "definition": "includes specific details or examples"}
    ]

print("Function defined: extract_qds_from_rubric")


Function defined: extract_qds_from_rubric


In [85]:
# Helper function: Grade a student answer using original rubric
def grade_with_rubric(student_answer, rubric_text, question_prompt):
    """
    Grade a student answer using the original rubric (general text).
    Returns 0 (does not meet) or 1 (meets expectation).
    """
    prompt = f"""You are grading a student answer using a rubric.

QUESTION: {question_prompt}

RUBRIC:
{rubric_text}

STUDENT ANSWER:
{student_answer}

Based STRICTLY on the rubric criteria, does this answer meet expectations?

You must output ONLY:
Grade: 0
OR
Grade: 1

Grade must be 0 (does not meet expectation) or 1 (meets expectation).
Be consistent in your interpretation of the rubric."""
    
    response = generate_with_delay(prompt, reasoning=False, delay=1.0)
    
    # Parse grade - look for explicit grade marking
    if "Grade: 1" in response or "Grade:1" in response:
        return 1
    elif "Grade: 0" in response or "Grade:0" in response:
        return 0
    # Fallback: check for keywords
    elif "meets expectation" in response.lower() and "does not meet" not in response.lower():
        return 1
    else:
        return 0

print("Function defined: grade_with_rubric")


Function defined: grade_with_rubric


In [86]:
# Helper function: Grade a student answer using QDs
def grade_with_qds(student_answer, qds, question_prompt):
    """
    Grade a student answer using quality dimensions (structured criteria).
    Returns 0 (does not meet) or 1 (meets expectation).
    """
    qd_descriptions = "\n".join([f"- {qd['name']}: {qd['definition']}" for qd in qds])
    
    prompt = f"""You are grading a student answer using structured quality dimensions (QDs).

QUESTION: {question_prompt}

QUALITY DIMENSIONS (Binary - Present or Absent):
{qd_descriptions}

STUDENT ANSWER:
{student_answer}

For each QD, determine if it is PRESENT (1) or ABSENT (0) in the answer.
If MOST QDs are present, the answer meets expectations (Grade: 1).
If MOST QDs are absent, the answer does not meet expectations (Grade: 0).

You must output ONLY:
Grade: 0
OR
Grade: 1

Be objective and consistent based on QD presence."""
    
    response = generate_with_delay(prompt, reasoning=False, delay=1.0)
    
    # Parse grade - look for explicit grade marking
    if "Grade: 1" in response or "Grade:1" in response:
        return 1
    elif "Grade: 0" in response or "Grade:0" in response:
        return 0
    # Fallback: check for keywords
    elif "meets expectation" in response.lower() and "does not meet" not in response.lower():
        return 1
    else:
        return 0

print("Function defined: grade_with_qds")


Function defined: grade_with_qds


In [87]:
# Helper function: Calculate flip rate for a single answer
def calculate_answer_flip_rate(grades):
    """
    Calculate flip rate for a single answer graded multiple times.
    Returns True if grades are inconsistent (flip occurred), False otherwise.
    """
    unique_grades = len(set(grades))
    return unique_grades > 1

print("Function defined: calculate_answer_flip_rate")


Function defined: calculate_answer_flip_rate


In [88]:
# Helper function: Analyze inconsistency patterns
def analyze_inconsistency(sample_answers, qd_results, current_qds):
    """
    Analyze which answers are flipping and why.
    Returns diagnostic information for refinement.
    """
    print("  Analyzing inconsistency patterns...")
    
    flip_cases = []
    for idx, result in qd_results.items():
        if result['has_flip']:
            flip_cases.append({
                'answer_idx': idx,
                'answer_text': sample_answers[idx][:150],
                'grades': result['grades']
            })
    
    if not flip_cases:
        print("  No flips detected")
        return None
    
    print(f"  Found {len(flip_cases)} answers with flips")
    return flip_cases

# Helper function: Refine QDs to reduce inconsistency
def refine_qds(current_qds, rubric_text, sample_answers, qd_results):
    """
    Ask LLM to refine QDs to reduce grading inconsistency.
    Uses MERGE/SPLIT/ADD/DROP operators based on analysis.
    Returns tuple: (refined_qds, operations_applied)
    """
    print("  Calling LLM to suggest QD refinements...")
    
    # Analyze what's causing flips
    flip_info = analyze_inconsistency(sample_answers, qd_results, current_qds)
    
    if not flip_info:
        print("  No inconsistency to fix")
        return current_qds, []
    
    qd_text = "\n".join([f"- {qd['name']}: {qd['definition']}" for qd in current_qds])
    flip_examples = "\n".join([f"{i+1}. Answer: {f['answer_text']}...\n   Grades across trials: {f['grades']}" 
                               for i, f in enumerate(flip_info[:3])])
    
    prompt = f"""You are refining Quality Dimensions (QDs) to reduce grading inconsistency.

ORIGINAL RUBRIC:
{rubric_text}

CURRENT QUALITY DIMENSIONS ({len(current_qds)} QDs):
{qd_text}

PROBLEM: These QDs cause inconsistent grading on some answers.

INCONSISTENT CASES:
{flip_examples}

GOAL: Make QDs more CONSISTENT, not more STRICT.

REFINEMENT OPERATORS TO CONSIDER:
1. MERGE: If two QDs always appear together, combine them into one compound QD
2. SPLIT: If a QD is too broad and covers multiple concepts, split into separate QDs  
3. ADD: If answers vary in ways not captured, add a missing QD
4. DROP: If a QD appears in all/no answers (doesn't discriminate), remove it

Suggest refined QDs that will reduce inconsistency while maintaining fair grading.
The refined QDs should be CLEARER and MORE OBJECTIVE, not stricter.

For EACH refined QD, indicate which operator you used by including "operator" field.

Output ONLY a valid JSON array with NO additional text:
[
  {{
    "name": "dimension_name",
    "definition": "clear, objective definition",
    "operator": "MERGE|SPLIT|ADD|DROP|KEEP",
    "explanation": "why this operator was chosen"
  }}
]"""
    
    response = generate_with_delay(prompt, reasoning=False, delay=2.0)
    
    # Parse JSON response
    try:
        import re
        json_match = re.search(r'\[.*\]', response, re.DOTALL)
        if json_match:
            qds = json.loads(json_match.group(0))
            if isinstance(qds, list) and len(qds) >= 2:
                print(f"  Successfully refined to {len(qds)} QDs")
                
                # Extract operations
                operations = []
                for qd in qds[:5]:
                    op = qd.get('operator', 'UNKNOWN')
                    explanation = qd.get('explanation', 'No explanation provided')
                    operations.append({
                        'qd_name': qd['name'],
                        'operator': op,
                        'explanation': explanation
                    })
                    print(f"    - {op}: {qd['name']}")
                
                return qds[:5], operations
    except Exception as e:
        print(f"  Failed to parse refined QDs: {e}")
        print("  Keeping current QDs")
        return current_qds, []
    
    return current_qds, []

print("Function defined: analyze_inconsistency, refine_qds")


Function defined: analyze_inconsistency, refine_qds


In [ ]:
# Main experiment function: Process a single (lab, question) pair
def process_question(lab_num, question_num, df_subset, rubric_text, question_prompt, num_trials=10, max_refinements=3):
    """
    Run the full experiment for one question:
    1. Baseline grading with rubric
    2. Extract QDs and grade with QDs
    3. Refine QDs if needed (up to max_refinements times)
    
    Returns dictionary with results.
    """
    print(f"\n{'='*80}")
    print(f"Processing Lab {lab_num}, Question {question_num}")
    print(f"Number of student answers: {len(df_subset)}")
    print(f"{'='*80}")
    
    # Get unique student answers
    student_answers = df_subset['question_text'].unique().tolist()
    print(f"Unique student texts: {len(student_answers)}")
    
    # Sample up to 5 answers for the experiment
    if len(student_answers) > 5:
        sample_answers = random.sample(student_answers, 5)
    else:
        sample_answers = student_answers
    
    print(f"Using {len(sample_answers)} answers for this experiment")
    
    # STEP 1: Baseline grading with original rubric
    print(f"\nSTEP 1: Baseline grading with original rubric ({num_trials} trials per answer)")
    baseline_results = {}
    baseline_flips = 0
    
    for idx, answer in enumerate(sample_answers):
        grades = []
        for trial in range(num_trials):
            grade = grade_with_rubric(answer, rubric_text, question_prompt)
            grades.append(grade)
        
        has_flip = calculate_answer_flip_rate(grades)
        baseline_results[idx] = {'grades': grades, 'has_flip': has_flip}
        if has_flip:
            baseline_flips += 1
        
        print(f"  Answer {idx+1}: grades={grades}, flip={has_flip}")
    
    baseline_flip_rate = (baseline_flips / len(sample_answers)) * 100
    print(f"Baseline flip rate: {baseline_flip_rate:.1f}% ({baseline_flips}/{len(sample_answers)} answers)")
    
    # STEP 2: Extract QDs from rubric
    print(f"\nSTEP 2: Extracting quality dimensions from rubric")
    current_qds = extract_qds_from_rubric(rubric_text)
    print(f"Extracted {len(current_qds)} QDs:")
    for qd in current_qds:
        print(f"  - {qd['name']}: {qd['definition']}")
    
    # STEP 3: Grade with QDs and refine if needed
    best_qds = current_qds
    best_qd_flip_rate = 100.0
    best_qd_results = None
    all_operations = []  # Track all operations applied
    
    for refinement_iter in range(max_refinements + 1):
        if refinement_iter == 0:
            print(f"\nSTEP 3.{refinement_iter}: Initial QD-based grading ({num_trials} trials per answer)")
        else:
            print(f"\nSTEP 3.{refinement_iter}: Refinement iteration {refinement_iter} ({num_trials} trials per answer)")
        
        qd_results = {}
        qd_flips = 0
        
        for idx, answer in enumerate(sample_answers):
            grades = []
            for trial in range(num_trials):
                grade = grade_with_qds(answer, current_qds, question_prompt)
                grades.append(grade)
            
            has_flip = calculate_answer_flip_rate(grades)
            qd_results[idx] = {'grades': grades, 'has_flip': has_flip}
            if has_flip:
                qd_flips += 1
            
            print(f"  Answer {idx+1}: grades={grades}, flip={has_flip}")
        
        qd_flip_rate = (qd_flips / len(sample_answers)) * 100
        print(f"QD flip rate: {qd_flip_rate:.1f}% ({qd_flips}/{len(sample_answers)} answers)")
        
        # Check if this is better
        if qd_flip_rate < best_qd_flip_rate:
            print(f"IMPROVEMENT: Flip rate reduced from {best_qd_flip_rate:.1f}% to {qd_flip_rate:.1f}%")
            best_qd_flip_rate = qd_flip_rate
            best_qds = current_qds
            best_qd_results = qd_results
            
            # If we've eliminated all flips, stop refining
            if qd_flip_rate == 0:
                print("Perfect consistency achieved! Stopping refinement.")
                break
        else:
            print(f"No improvement: {qd_flip_rate:.1f}% >= {best_qd_flip_rate:.1f}%")
        
        # Refine QDs if we haven't reached max iterations and still have flips
        if refinement_iter < max_refinements and qd_flip_rate > 0:
            print(f"\nRefining QDs (attempt {refinement_iter + 1}/{max_refinements})...")
            current_qds, operations = refine_qds(current_qds, rubric_text, sample_answers, qd_results)
            if operations:
                all_operations.append({
                    'iteration': refinement_iter + 1,
                    'operations': operations
                })
            print(f"Refined to {len(current_qds)} QDs:")
            for qd in current_qds:
                print(f"  - {qd['name']}: {qd['definition']}")
        else:
            break
    
    # Calculate improvement
    improvement = baseline_flip_rate - best_qd_flip_rate
    
    print(f"\n{'='*80}")
    print(f"FINAL RESULTS for Lab {lab_num}, Question {question_num}:")
    print(f"  Baseline flip rate: {baseline_flip_rate:.1f}%")
    print(f"  Best QD flip rate: {best_qd_flip_rate:.1f}%")
    print(f"  Improvement: {improvement:+.1f} percentage points")
    
    # Print operations summary
    if all_operations:
        print(f"\nOPERATIONS APPLIED:")
        for iter_ops in all_operations:
            print(f"  Iteration {iter_ops['iteration']}:")
            for op in iter_ops['operations']:
                print(f"    - {op['operator']}: {op['qd_name']}")
                if op.get('explanation'):
                    print(f"      Reason: {op['explanation'][:100]}...")
    else:
        print(f"\nOPERATIONS APPLIED: None (initial QDs were sufficient)")
    
    print(f"{'='*80}")
    
    return {
        'lab_number': lab_num,
        'question_number': question_num,
        'num_answers': len(sample_answers),
        'baseline_flip_rate': baseline_flip_rate,
        'qd_flip_rate': best_qd_flip_rate,
        'improvement': improvement,
        'qds_used': best_qds,
        'operations_applied': all_operations,  # NEW: track operations
        'baseline_results': baseline_results,
        'qd_results': best_qd_results
    }

print("Function defined: process_question")


Function defined: process_question


In [90]:
# Run experiment on all questions
def run_full_experiment(df, rubrics_dict, num_trials=10, max_refinements=3):
    """
    Run the experiment on all (lab, question) pairs in the dataframe.
    """
    print(f"\n{'#'*80}")
    print(f"# STARTING FULL EXPERIMENT")
    print(f"# Number of trials per answer: {num_trials}")
    print(f"# Max refinement iterations: {max_refinements}")
    print(f"{'#'*80}")
    
    all_results = []
    
    # Get unique (lab, question) pairs
    lab_question_pairs = df[['lab_number', 'question_number']].drop_duplicates().values.tolist()
    print(f"\nTotal (lab, question) pairs to process: {len(lab_question_pairs)}")
    
    for lab_num, question_num in lab_question_pairs:
        # Get subset for this question
        df_subset = df[(df['lab_number'] == lab_num) & (df['question_number'] == question_num)]
        
        # Get rubric
        rubric_text = rubrics_dict.get((lab_num, question_num), '')
        if not rubric_text:
            print(f"\nSkipping Lab {lab_num}, Question {question_num}: No rubric found")
            continue
        
        # Get question prompt (use first row's context)
        question_prompt = f"Lab {lab_num}, Question {question_num}"
        
        # Process this question
        result = process_question(
            lab_num, 
            question_num, 
            df_subset, 
            rubric_text, 
            question_prompt,
            num_trials=num_trials,
            max_refinements=max_refinements
        )
        
        all_results.append(result)
    
    # Summary statistics
    print(f"\n{'#'*80}")
    print(f"# EXPERIMENT COMPLETE")
    print(f"{'#'*80}")
    print(f"\nProcessed {len(all_results)} questions")
    
    avg_baseline = sum(r['baseline_flip_rate'] for r in all_results) / len(all_results)
    avg_qd = sum(r['qd_flip_rate'] for r in all_results) / len(all_results)
    avg_improvement = sum(r['improvement'] for r in all_results) / len(all_results)
    
    print(f"\nAVERAGE RESULTS:")
    print(f"  Average baseline flip rate: {avg_baseline:.1f}%")
    print(f"  Average QD flip rate: {avg_qd:.1f}%")
    print(f"  Average improvement: {avg_improvement:+.1f} percentage points")
    
    # Count how many improved
    improved_count = sum(1 for r in all_results if r['improvement'] > 0)
    print(f"\nQuestions with improvement: {improved_count}/{len(all_results)} ({100*improved_count/len(all_results):.1f}%)")
    
    return all_results

print("Function defined: run_full_experiment")


Function defined: run_full_experiment


## Approach Explanation

**Goal**: Demonstrate that using Quality Dimensions (QDs) leads to more consistent grading than using general rubrics.

**Method**:
1. **Baseline**: Grade each answer 10 times using the ORIGINAL RUBRIC (general text)
   - Calculate flip rate: % of answers that get inconsistent grades
2. **QD Extraction**: Extract binary quality dimensions from the rubric
3. **QD Grading**: Grade same answers 10 times using the EXTRACTED QDs (structured criteria)
   - Calculate flip rate with QD-based grading
4. **Refinement** (if needed): If QD flip rate is still high:
   - Analyze WHY inconsistency occurs
   - Apply MERGE/SPLIT/ADD/DROP operators to refine QDs
   - Re-grade and check if consistency improved
   - Repeat up to 3 times

**Key Insight**: QDs should be MORE CONSISTENT, not MORE STRICT. We're testing if structured quality dimensions reduce grading variance compared to general rubric text.


## Run Experiment

Now we can run the experiment. Start with a small test (e.g., 1-2 questions) to verify everything works before running on all questions.

Set parameters:
- `num_trials`: Number of times to grade each answer (default 10)
- `max_refinements`: Maximum number of QD refinement attempts (default 3)


In [91]:
# TEST RUN: Run on first question only to verify setup
# Uncomment to test:
import random
from datetime import datetime
from time import sleep
import time
import json
test_df = initial_df[(initial_df['lab_number'] == 1) & (initial_df['question_number'] == 1)]
print(f"Test dataframe size: {len(test_df)}")

result = process_question(
    lab_num=1,
    question_num=1,
    df_subset=test_df,
    rubric_text=rubrics_dict.get((1, 1), ''),
    question_prompt="Lab 1, Question 1",
    num_trials=3,  # Reduced for testing
    max_refinements=2  # Reduced for testing
)

print("\nTest complete! Review output above before running full experiment.")


Test dataframe size: 46

Processing Lab 1, Question 1
Number of student answers: 46
Unique student texts: 12
Using 5 answers for this experiment

STEP 1: Baseline grading with original rubric (3 trials per answer)
  Answer 1: grades=[0, 1, 0], flip=True
  Answer 2: grades=[1, 1, 1], flip=False
  Answer 3: grades=[1, 1, 1], flip=False
  Answer 4: grades=[1, 1, 1], flip=False
  Answer 5: grades=[1, 1, 1], flip=False
Baseline flip rate: 20.0% (1/5 answers)

STEP 2: Extracting quality dimensions from rubric
  Calling LLM to extract QDs from rubric...
  Successfully extracted 4 QDs
Extracted 4 QDs:
  - IdentifiesHAL: The response explicitly states that HAL stands for “Hardware Abstraction Layer.”
  - ExplainsSimplificationPurpose: The response correctly explains that the HAL is used to simplify interaction with the hardware.
  - ProvidesAdditionalCorrectDetails: The response includes extra, accurate details describing why having a HAL is useful (e.g., portability, modularity, easier driver 

In [106]:
test_df

,username,lab_number,question_number,ts,question_text,grade,term,rubric
588,c6shao,1,1,2025-05-16_11-32-09,HAL stands for Hardware Abstraction Layer. It ...,1,s25,Meets Expectation: Students correctly identify...
589,c6shao,1,1,2025-05-17_14-00-47,HAL stands for Hardware Abstraction Layer. It ...,1,s25,Meets Expectation: Students correctly identify...
590,c6shao,1,1,2025-05-19_00-42-23,HAL stands for Hardware Abstraction Layer. It ...,1,s25,Meets Expectation: Students correctly identify...
591,c6shao,1,1,2025-05-28_11-45-17,HAL stands for Hardware Abstraction Layer. It ...,0,s25,Meets Expectation: Students correctly identify...
592,c6shao,1,1,2025-05-28_11-45-17,HAL stands for Hardware Abstraction Layer. It ...,0,s25,Meets Expectation: Students correctly identify...
593,c6shao,1,1,2025-05-28_11-45-17,HAL stands for Hardware Abstraction Layer. It ...,0,s25,Meets Expectation: Students correctly identify...
640,g223zhan,1,1,2025-05-16_18-23-00,The HAL is a set of software functions that ma...,1,s25,Meets Expectation: Students correctly identify...
641,g223zhan,1,1,2025-05-16_18-34-22,The HAL is a set of software functions that ma...,0,s25,Meets Expectation: Students correctly identify...
758,j3hou,1,1,2025-05-14_15-26-33,The HAL (Hardware Abstraction Layer) is a blac...,1,s25,Meets Expectation: Students correctly identify...
759,j3hou,1,1,2025-05-21_22-28-24,The HAL (Hardware Abstraction Layer) is a blac...,1,s25,Meets Expectation: Students correctly identify...


In [98]:
# View operations applied (if any)
print("\n" + "="*80)
print("OPERATIONS TRACKING")
print("="*80)

if 'operations_applied' in result and result['operations_applied']:
    print(f"\nTotal refinement iterations: {len(result['operations_applied'])}\n")
    
    for iter_ops in result['operations_applied']:
        print(f"Iteration {iter_ops['iteration']}:")
        print("-" * 60)
        
        for op in iter_ops['operations']:
            print(f"\nQD: {op['qd_name']}")
            print(f"  Operator: {op['operator']}")
            print(f"  Explanation: {op['explanation']}")
else:
    print("\nNo operations were applied.")
    print("Reason: Initial QDs achieved perfect consistency (0% flip rate)")
    print("\nThis is actually good! It means:")
    print("  - The extracted QDs were already well-defined")
    print("  - No refinement was needed")
    print("  - QD-based grading was immediately more consistent than rubric-based")
    
print("\n" + "="*80)



OPERATIONS TRACKING

Total refinement iterations: 1

Iteration 1:
------------------------------------------------------------

QD: ExpandsAcronym
  Operator: SPLIT
  Explanation: The original IdentifiesHAL combined two ideas (knowing the concept and spelling out the acronym). Splitting isolates the literal expansion, making grading of missing expansions consistent.

QD: DescribesHALConcept
  Operator: SPLIT
  Explanation: Separating the conceptual description from the acronym expansion lets graders award credit when the student understands HAL even if they omit the full form, reducing disagreement.

QD: ExplainsSimplificationAndBenefit
  Operator: MERGE
  Explanation: The original two dimensions (purpose explanation and extra details) were often judged inconsistently when a student gave only one of them. Merging them into a single, objective requirement that both purpose and a benefit be present creates a clearer pass/fail criterion.

QD: NoFactualErrors
  Operator: KEEP
  Explanatio

In [100]:
# Comprehensive visualization of results
def visualize_experiment_results(result):
    """
    Create a publication-ready visualization of the experiment results.
    Shows: flip rate comparison, per-answer grades, operations applied, and QDs used.
    """
    print("\n" + "="*100)
    print(f"EXPERIMENT RESULTS: Lab {result['lab_number']}, Question {result['question_number']}")
    print("="*100)
    
    # Section 1: Overall Metrics
    print("\n" + "-"*100)
    print("1. OVERALL FLIP RATE COMPARISON")
    print("-"*100)
    
    print(f"\n{'Metric':<40} {'Baseline':<20} {'QD-Based':<20} {'Change':<20}")
    print("-"*100)
    print(f"{'Flip Rate':<40} {result['baseline_flip_rate']:>6.1f}% {result['qd_flip_rate']:>18.1f}% {result['improvement']:>14.1f}pp")
    print(f"{'Consistency Rate':<40} {100-result['baseline_flip_rate']:>6.1f}% {100-result['qd_flip_rate']:>18.1f}% {-(result['improvement']):>14.1f}pp")
    print(f"{'Number of Answers':<40} {result['num_answers']:>6} {result['num_answers']:>18} {'--':<20}")
    
    # Section 2: Per-Answer Grading
    print("\n" + "-"*100)
    print("2. PER-ANSWER GRADING RESULTS")
    print("-"*100)
    
    print(f"\n{'Answer':<10} {'Baseline Grades':<25} {'QD Grades':<25} {'Baseline Status':<20} {'QD Status':<20} {'Change':<15}")
    print("-"*100)
    
    for idx in sorted(result['baseline_results'].keys()):
        base_grades = result['baseline_results'][idx]['grades']
        qd_grades = result['qd_results'][idx]['grades']
        base_flip = result['baseline_results'][idx]['has_flip']
        qd_flip = result['qd_results'][idx]['has_flip']
        
        base_str = str(base_grades)
        qd_str = str(qd_grades)
        
        base_status = "FLIP" if base_flip else "Consistent"
        qd_status = "FLIP" if qd_flip else "Consistent"
        
        if base_flip and not qd_flip:
            change = "FIXED"
        elif not base_flip and not qd_flip:
            # Check if grades changed
            if base_grades == qd_grades:
                change = "Same"
            else:
                change = "Grade changed"
        elif base_flip and qd_flip:
            change = "Still flips"
        else:
            change = "Worsened"
        
        print(f"#{idx+1:<9} {base_str:<25} {qd_str:<25} {base_status:<20} {qd_status:<20} {change:<15}")
    
    # Section 3: Operations Applied
    print("\n" + "-"*100)
    print("3. REFINEMENT OPERATIONS APPLIED")
    print("-"*100)
    
    if 'operations_applied' in result and result['operations_applied']:
        for iter_ops in result['operations_applied']:
            print(f"\nRefinement Iteration {iter_ops['iteration']}:")
            print("-"*100)
            
            # Count operators
            op_counts = {}
            for op in iter_ops['operations']:
                op_type = op['operator']
                op_counts[op_type] = op_counts.get(op_type, 0) + 1
            
            print(f"\nOperator Summary: {', '.join([f'{k}: {v}' for k, v in sorted(op_counts.items())])}")
            print()
            
            for i, op in enumerate(iter_ops['operations'], 1):
                print(f"{i}. QD: {op['qd_name']}")
                print(f"   Operator: {op['operator']}")
                print(f"   Rationale: {op['explanation']}")
                print()
    else:
        print("\nNo operations applied - initial QDs were sufficient.")
    
    # Section 4: Final Quality Dimensions
    print("-"*100)
    print("4. FINAL QUALITY DIMENSIONS USED")
    print("-"*100)
    
    print(f"\nTotal QDs: {len(result['qds_used'])}\n")
    
    for i, qd in enumerate(result['qds_used'], 1):
        operator = qd.get('operator', 'N/A')
        print(f"{i}. {qd['name']} [{operator}]")
        print(f"   Definition: {qd['definition']}")
        if 'explanation' in qd:
            print(f"   Why {operator}: {qd['explanation'][:150]}...")
        print()
    
    # Section 5: Summary Statistics
    print("-"*100)
    print("5. SUMMARY STATISTICS")
    print("-"*100)
    
    # Calculate grade changes
    grade_changes = 0
    flip_fixes = 0
    for idx in result['baseline_results'].keys():
        if result['baseline_results'][idx]['has_flip'] and not result['qd_results'][idx]['has_flip']:
            flip_fixes += 1
        if result['baseline_results'][idx]['grades'] != result['qd_results'][idx]['grades']:
            grade_changes += 1
    
    print(f"\nFlips fixed: {flip_fixes}/{result['num_answers']}")
    print(f"Answers with grade changes: {grade_changes}/{result['num_answers']}")
    print(f"Improvement: {result['improvement']:+.1f} percentage points")
    
    if result['qd_flip_rate'] == 0:
        print("\nRESULT: Perfect consistency achieved!")
    elif result['qd_flip_rate'] < result['baseline_flip_rate']:
        print(f"\nRESULT: Flip rate reduced by {result['improvement']:.1f}pp")
    else:
        print("\nRESULT: No improvement in flip rate")
    
    print("\n" + "="*100 + "\n")

print("Function defined: visualize_experiment_results")


Function defined: visualize_experiment_results


In [101]:
# Visualize the result with your data
visualize_experiment_results(result)



EXPERIMENT RESULTS: Lab 1, Question 1

----------------------------------------------------------------------------------------------------
1. OVERALL FLIP RATE COMPARISON
----------------------------------------------------------------------------------------------------

Metric                                   Baseline             QD-Based             Change              
----------------------------------------------------------------------------------------------------
Flip Rate                                  20.0%                0.0%           20.0pp
Consistency Rate                           80.0%              100.0%          -20.0pp
Number of Answers                             5                  5 --                  

----------------------------------------------------------------------------------------------------
2. PER-ANSWER GRADING RESULTS
----------------------------------------------------------------------------------------------------

Answer     Baseline Grades

In [103]:
# Test on multiple specific questions
def test_multiple_questions(lab_question_pairs, num_trials=3, max_refinements=2):
    """
    Test the experiment on specific (lab, question) pairs.
    
    Args:
        lab_question_pairs: List of tuples [(lab_num, question_num), ...]
        num_trials: Number of grading trials per answer
        max_refinements: Maximum refinement iterations
    
    Returns:
        List of results dictionaries
    """
    results = []
    
    print(f"\n{'#'*100}")
    print(f"# TESTING {len(lab_question_pairs)} QUESTIONS")
    print(f"# Trials per answer: {num_trials}")
    print(f"# Max refinements: {max_refinements}")
    print(f"{'#'*100}\n")
    
    for lab_num, question_num in lab_question_pairs:
        # Get subset for this question
        df_subset = initial_df[(initial_df['lab_number'] == lab_num) & 
                                (initial_df['question_number'] == question_num)]
        
        # Get rubric
        rubric_text = rubrics_dict.get((lab_num, question_num), '')
        if not rubric_text:
            print(f"Skipping Lab {lab_num}, Question {question_num}: No rubric found\n")
            continue
        
        # Get question prompt
        question_prompt = f"Lab {lab_num}, Question {question_num}"
        
        # Process this question
        result = process_question(
            lab_num, 
            question_num, 
            df_subset, 
            rubric_text, 
            question_prompt,
            num_trials=num_trials,
            max_refinements=max_refinements
        )
        
        results.append(result)
    
    # Summary
    print(f"\n{'#'*100}")
    print(f"# TEST COMPLETE: {len(results)} questions processed")
    print(f"{'#'*100}\n")
    
    # Quick summary table
    print(f"{'Lab':<6} {'Q#':<6} {'Baseline %':<12} {'QD %':<12} {'Improvement':<15} {'Operations':<20}")
    print("-"*100)
    
    for r in results:
        ops_count = len(r['operations_applied'][0]['operations']) if r.get('operations_applied') else 0
        ops_summary = f"{ops_count} ops" if ops_count > 0 else "None"
        
        print(f"{r['lab_number']:<6} {r['question_number']:<6} {r['baseline_flip_rate']:>10.1f}% "
              f"{r['qd_flip_rate']:>10.1f}% {r['improvement']:>12.1f}pp  {ops_summary:<20}")
    
    # Overall stats
    if results:
        avg_baseline = sum(r['baseline_flip_rate'] for r in results) / len(results)
        avg_qd = sum(r['qd_flip_rate'] for r in results) / len(results)
        avg_improvement = sum(r['improvement'] for r in results) / len(results)
        
        print("-"*100)
        print(f"{'AVERAGE':<6} {'':>6} {avg_baseline:>10.1f}% {avg_qd:>10.1f}% {avg_improvement:>12.1f}pp")
        print("\n")
    
    return results

print("Function defined: test_multiple_questions")


Function defined: test_multiple_questions


In [104]:
# Check what questions are available
print("Available (Lab, Question) pairs with data:\n")
available_pairs = initial_df[['lab_number', 'question_number']].drop_duplicates().sort_values(['lab_number', 'question_number'])

for lab in range(1, 6):
    lab_questions = available_pairs[available_pairs['lab_number'] == lab]['question_number'].tolist()
    if lab_questions:
        print(f"Lab {lab}: Questions {lab_questions}")

print(f"\nTotal available: {len(available_pairs)} (lab, question) pairs")


Available (Lab, Question) pairs with data:

Lab 1: Questions [1, 2, 3, 4, 5]
Lab 2: Questions [1, 2, 3, 4, 5]
Lab 3: Questions [1, 2, 3, 4, 5]
Lab 4: Questions [1, 2, 3, 4, 5]
Lab 5: Questions [1, 2, 3, 4, 5]

Total available: 25 (lab, question) pairs


In [105]:
test_results = test_multiple_questions([
    (1, 1),
    (1, 2),
    (1, 3)
], num_trials=5, max_refinements=2)



####################################################################################################
# TESTING 3 QUESTIONS
# Trials per answer: 5
# Max refinements: 2
####################################################################################################


Processing Lab 1, Question 1
Number of student answers: 46
Unique student texts: 12
Using 5 answers for this experiment

STEP 1: Baseline grading with original rubric (5 trials per answer)
Connection Error: HTTPConnectionPool(host='ece-nebula16.eng.uwaterloo.ca', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("<urllib3.connection.HTTPConnection object at 0x3009eeb70>: Failed to establish a new connection: [Errno 49] Can't assign requested address"))
Connection Error: HTTPConnectionPool(host='ece-nebula16.eng.uwaterloo.ca', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("<urllib3.connection.HTTPConnection object at 0x3009ef0e0>: Failed to e

In [ ]:
# EXAMPLE 2: Test one question from each lab
# Uncomment to run:

# test_results = test_multiple_questions([
#     (1, 1),
#     (2, 1),
#     (3, 1),
#     (4, 1),
#     (5, 1)
# ], num_trials=3, max_refinements=2)


In [ ]:
# EXAMPLE 3: Visualize a specific result from your test
# Uncomment to run (after running one of the examples above):

# visualize_experiment_results(test_results[0])  # First result
# visualize_experiment_results(test_results[1])  # Second result
# etc.


In [ ]:
# Save results to JSON for later analysis
# Uncomment to run:

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# output_file = f"test_results_{timestamp}.json"
# 
# with open(output_file, 'w') as f:
#     json.dump(test_results, f, indent=2)
# 
# print(f"Results saved to: {output_file}")


## Understanding Operations

When refinement is needed, the system will apply one or more of these operators:

| Operator | Purpose | When Applied | Example |
|----------|---------|--------------|---------|
| **KEEP** | No change | QD is working well | Keep as-is |
| **MERGE** | Combine coupled QDs | Two QDs always appear together | Merge "mentions_energy" + "mentions_charge" → "energy_per_charge" |
| **SPLIT** | Break apart broad QD | QD covers multiple concepts | Split "explains_pointers" → "address_concept" + "operators" + "use_cases" |
| **ADD** | Introduce new QD | Missing aspect causing flips | Add "provides_example" if examples help consistency |
| **DROP** | Remove QD | QD doesn't discriminate (appears in all/no answers) | Drop "mentions_voltage" if 98% of answers have it |

### In Your Test Run:

- **Baseline**: 20% flip rate (1 out of 5 answers had inconsistent grades)
- **QD Extraction**: Successfully extracted 4 QDs from rubric
- **QD Grading**: 0% flip rate (all 5 answers graded consistently)
- **Operations**: **None needed** - immediate success!

This means the initial QDs were already well-defined and objective enough to eliminate grading inconsistency.


In [ ]:
result

{'lab_number': 1,
 'question_number': 1,
 'num_answers': 5,
 'baseline_flip_rate': 20.0,
 'qd_flip_rate': 0.0,
 'improvement': 20.0,
 'qds_used': [{'name': 'ExpandsAcronym',
   'definition': 'The response explicitly states that HAL stands for “Hardware Abstraction Layer.”',
   'operator': 'SPLIT',
   'explanation': 'The original IdentifiesHAL combined two ideas (knowing the concept and spelling out the acronym). Splitting isolates the literal expansion, making grading of missing expansions consistent.'},
  {'name': 'DescribesHALConcept',
   'definition': 'The response indicates that HAL is a software layer that abstracts hardware functionality (e.g., provides a uniform API, hides low‑level details).',
   'operator': 'SPLIT',
   'explanation': 'Separating the conceptual description from the acronym expansion lets graders award credit when the student understands HAL even if they omit the full form, reducing disagreement.'},
  {'name': 'ExplainsSimplificationAndBenefit',
   'definition':

In [94]:
# FULL EXPERIMENT: Run on all questions
# WARNING: This will take a long time (10 trials x ~25 questions x multiple refinements)

# experiment_results = run_full_experiment(
#     df=initial_df,
#     rubrics_dict=rubrics_dict,
#     num_trials=10,
#     max_refinements=3
# )
# 
# # Save results to JSON
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# output_file = f"qd_refinement_experiment_{timestamp}.json"
# with open(output_file, 'w') as f:
#     json.dump(experiment_results, f, indent=2)
# 
# print(f"\nResults saved to: {output_file}")


## Analysis Functions

Functions to analyze and visualize the results after the experiment completes.


In [95]:
# Analysis function: Create summary report
def create_summary_report(results):
    """
    Create a detailed summary report from experiment results.
    """
    print(f"\n{'='*80}")
    print(f"DETAILED SUMMARY REPORT")
    print(f"{'='*80}\n")
    
    # Per-question breakdown
    print(f"{'Lab':>4} {'Q#':>3} {'Answers':>8} {'Baseline%':>10} {'QD%':>10} {'Change':>10}")
    print(f"{'-'*60}")
    
    for r in results:
        print(f"{r['lab_number']:>4} {r['question_number']:>3} {r['num_answers']:>8} "
              f"{r['baseline_flip_rate']:>9.1f}% {r['qd_flip_rate']:>9.1f}% {r['improvement']:>+9.1f}pp")
    
    print(f"\n{'='*80}")
    
    # Statistics by lab
    print(f"\nRESULTS BY LAB:")
    for lab_num in range(1, 6):
        lab_results = [r for r in results if r['lab_number'] == lab_num]
        if lab_results:
            avg_baseline = sum(r['baseline_flip_rate'] for r in lab_results) / len(lab_results)
            avg_qd = sum(r['qd_flip_rate'] for r in lab_results) / len(lab_results)
            avg_improvement = sum(r['improvement'] for r in lab_results) / len(lab_results)
            print(f"  Lab {lab_num}: Baseline={avg_baseline:.1f}%, QD={avg_qd:.1f}%, Improvement={avg_improvement:+.1f}pp")
    
    # Overall statistics
    print(f"\nOVERALL STATISTICS:")
    total = len(results)
    improved = sum(1 for r in results if r['improvement'] > 0)
    worsened = sum(1 for r in results if r['improvement'] < 0)
    unchanged = sum(1 for r in results if r['improvement'] == 0)
    
    print(f"  Total questions: {total}")
    print(f"  Improved: {improved} ({100*improved/total:.1f}%)")
    print(f"  Worsened: {worsened} ({100*worsened/total:.1f}%)")
    print(f"  Unchanged: {unchanged} ({100*unchanged/total:.1f}%)")
    
    avg_baseline = sum(r['baseline_flip_rate'] for r in results) / total
    avg_qd = sum(r['qd_flip_rate'] for r in results) / total
    avg_improvement = sum(r['improvement'] for r in results) / total
    
    print(f"\n  Average baseline flip rate: {avg_baseline:.1f}%")
    print(f"  Average QD flip rate: {avg_qd:.1f}%")
    print(f"  Average improvement: {avg_improvement:+.1f} percentage points")
    
    print(f"\n{'='*80}\n")

print("Function defined: create_summary_report")


Function defined: create_summary_report


In [96]:
# Visualize comparison: Rubric-based vs QD-based grading
def visualize_results(result):
    """
    Create a clear visualization comparing rubric-based vs QD-based grading.
    """
    print(f"\n{'='*80}")
    print(f"GRADING COMPARISON: Lab {result['lab_number']}, Question {result['question_number']}")
    print(f"{'='*80}\n")
    
    print(f"Number of student answers tested: {result['num_answers']}")
    print(f"Trials per answer: {len(list(result['baseline_results'].values())[0]['grades'])}\n")
    
    print(f"{'Method':<30} {'Flip Rate':<15} {'Consistency':<15}")
    print(f"{'-'*60}")
    print(f"{'Original Rubric (General)':<30} {result['baseline_flip_rate']:>6.1f}%  {100-result['baseline_flip_rate']:>12.1f}%")
    print(f"{'Quality Dimensions (Refined)':<30} {result['qd_flip_rate']:>6.1f}%  {100-result['qd_flip_rate']:>12.1f}%")
    print(f"{'-'*60}")
    print(f"{'Improvement':<30} {result['improvement']:>+6.1f}pp")
    
    print(f"\n{'='*80}\n")
    
    # Show which answers flipped in each method
    print("PER-ANSWER BREAKDOWN:")
    print(f"{'Answer':<10} {'Rubric Grades':<30} {'QD Grades':<30} {'Improvement':<15}")
    print(f"{'-'*80}")
    
    for idx in result['baseline_results'].keys():
        base_grades = result['baseline_results'][idx]['grades']
        qd_grades = result['qd_results'][idx]['grades']
        base_flip = result['baseline_results'][idx]['has_flip']
        qd_flip = result['qd_results'][idx]['has_flip']
        
        base_str = str(base_grades) + (" FLIP" if base_flip else "")
        qd_str = str(qd_grades) + (" FLIP" if qd_flip else "")
        
        if base_flip and not qd_flip:
            improvement = "FIXED"
        elif not base_flip and not qd_flip:
            improvement = "Consistent"
        elif base_flip and qd_flip:
            improvement = "Still flips"
        else:
            improvement = "Worsened"
        
        print(f"{idx+1:<10} {base_str:<30} {qd_str:<30} {improvement:<15}")
    
    print(f"\n{'='*80}\n")

print("Function defined: visualize_results")


Function defined: visualize_results


In [ ]:
# Example: Load and analyze saved results
# Uncomment to use after experiment completes:

# with open('qd_refinement_experiment_YYYYMMDD_HHMMSS.json', 'r') as f:
#     loaded_results = json.load(f)

# create_summary_report(loaded_results)


## Summary of Implementation

### What This Code Does

This notebook implements an experiment to test if **Quality Dimensions (QDs)** lead to more consistent grading than general rubric text.

### Key Functions

1. **`extract_qds_from_rubric(rubric_text)`** - Extracts binary quality dimensions from rubric
2. **`grade_with_rubric(answer, rubric, prompt)`** - Grades using original rubric text
3. **`grade_with_qds(answer, qds, prompt)`** - Grades using structured QDs  
4. **`refine_qds(qds, rubric, answers, results)`** - Uses MERGE/SPLIT/ADD/DROP to refine QDs
5. **`process_question(...)`** - Main experiment loop for one question
6. **`run_full_experiment(...)`** - Runs experiment on all questions
7. **`visualize_results(result)`** - Shows comparison between rubric vs QD grading

### Alignment with Research

- **Baseline**: General rubric text (current approach)
- **Treatment**: Structured QDs (proposed approach)
- **Metric**: Flip rate reduction (consistency improvement)
- **Refinement**: MERGE/SPLIT/ADD/DROP operators from Section 4 of Operations.md
- **Goal**: Show QDs are MORE CONSISTENT, not more strict

### Important Points

- QDs should make grading **consistent**, not **harder**
- Flip rate = % of answers with grade disagreements across trials
- Lower flip rate = more consistent grading = better
- Refinement focuses on making QDs more objective and distinguishable
